# Lesson 13a: Multi-Agent Reinforcement Learning Theory

**Learning Objectives:**
- Understand multi-agent systems and game theory foundations
- Learn Nash equilibria and solution concepts
- Master independent Q-learning and centralized training
- Study cooperative, competitive, and mixed settings
- Explore communication and coordination mechanisms
- Understand emergent behaviors and social dilemmas

**Prerequisites:** MDPs (1a), Q-Learning (4a), Policy Gradients (8a)

**References:**
- Shoham & Leyton-Brown - Multiagent Systems
- Lowe et al. (2017) - MADDPG
- Foerster et al. (2018) - COMA, QMIX
- OpenAI Five (2019), AlphaStar (2019)

## 1. Introduction to Multi-Agent Systems

### 1.1 What is Multi-Agent RL?

**Single-agent RL:** One agent learning in environment
$$\max_\pi \mathbb{E}[\sum_t \gamma^t r_t]$$

**Multi-agent RL:** Multiple agents learning simultaneously
- Each agent $i$ has policy $\pi_i$
- Environment responds to joint actions
- Rewards may be individual or shared

### 1.2 Why Multi-Agent?

**Real-world applications:**
1. **Robotics**: Multiple robots coordinating
2. **Game AI**: Chess, Go, Poker, Dota, StarCraft
3. **Autonomous vehicles**: Traffic coordination
4. **Economics**: Market modeling, auctions
5. **Social simulation**: Crowd behavior

**Key challenges:**
- Non-stationarity: Other agents are learning too!
- Credit assignment: Which agent caused outcome?
- Scalability: Grows exponentially with agents
- Communication: How should agents coordinate?

### 1.3 Types of Multi-Agent Settings

**Cooperative (team):**
- Shared reward: $r_i = r$ for all agents
- Goal: Maximize team performance
- Examples: Robot teams, multiplayer co-op games

**Competitive (adversarial):**
- Zero-sum: $\sum_i r_i = 0$
- Goal: Beat opponents
- Examples: Chess, Go, Poker

**Mixed (general-sum):**
- Individual rewards: $r_i \neq r_j$
- Partial cooperation/competition
- Examples: Autonomous driving, economics

## 2. Game Theory Foundations

### 2.1 Normal Form Games

**Game definition:**
- Players: $\mathcal{N} = \{1, 2, ..., n\}$
- Actions: $\mathcal{A}_i$ for each player $i$
- Payoffs: $u_i: \mathcal{A}_1 \times ... \times \mathcal{A}_n \to \mathbb{R}$

**Example: Prisoner's Dilemma**
```
           Cooperate  Defect
Cooperate    (3,3)    (0,5)
Defect       (5,0)    (1,1)
```

### 2.2 Nash Equilibrium

**Definition:** Policy profile $(\pi_1^*, ..., \pi_n^*)$ where no agent can improve by deviating:

$$u_i(\pi_i^*, \pi_{-i}^*) \geq u_i(\pi_i, \pi_{-i}^*) \quad \forall i, \pi_i$$

where $\pi_{-i}$ denotes policies of all agents except $i$

**Properties:**
- Every finite game has at least one Nash equilibrium (possibly mixed)
- Not necessarily unique
- Not necessarily optimal (Pareto efficient)

**Prisoner's Dilemma equilibrium:**
- Nash: (Defect, Defect) - payoff (1,1)
- Optimal: (Cooperate, Cooperate) - payoff (3,3)
- Social dilemma: Individual rationality ≠ collective rationality

### 2.3 Zero-Sum Games

**Two-player zero-sum:** $u_1 + u_2 = 0$

**Minimax theorem:** Nash equilibrium = minimax solution
$$\max_{\pi_1} \min_{\pi_2} u_1(\pi_1, \pi_2) = \min_{\pi_2} \max_{\pi_1} u_1(\pi_1, \pi_2)$$

**Examples:**
- Chess, Go, Poker
- Two-player Atari (e.g., Pong)
- Adversarial training (GANs)

### 2.4 Stochastic Games (Markov Games)

**Extension of MDPs to multiple agents:**

- States: $s \in \mathcal{S}$
- Joint actions: $\mathbf{a} = (a_1, ..., a_n) \in \mathcal{A}_1 \times ... \times \mathcal{A}_n$
- Transition: $p(s'|s, \mathbf{a})$
- Rewards: $r_i(s, \mathbf{a})$ for each agent $i$

**Value functions:**
$$V_i^{\pi_1, ..., \pi_n}(s) = \mathbb{E}_{\pi_1,...,\pi_n}\left[\sum_{t=0}^\infty \gamma^t r_i(s_t, \mathbf{a}_t) \mid s_0 = s\right]$$

**Nash equilibrium:** Policies where no agent can improve given others' policies

## 3. Independent Learning

### 3.1 Independent Q-Learning (IQL)

**Idea:** Each agent treats others as part of environment

**Agent $i$'s update:**
$$Q_i(s, a_i) \leftarrow Q_i(s, a_i) + \alpha[r_i + \gamma \max_{a_i'} Q_i(s', a_i') - Q_i(s, a_i)]$$

**Observations:**
- Agent $i$ only sees own actions $a_i$ (not joint action $\mathbf{a}$)
- Learns $Q_i(s, a_i)$ not $Q_i(s, \mathbf{a})$

**Properties:**
- Simple to implement
- Scales well (each agent independent)
- **Problem:** Environment is non-stationary!
  - Other agents' policies changing
  - No convergence guarantees
  - Can lead to suboptimal equilibria

### 3.2 Non-Stationarity Problem

**Single-agent RL assumption:** Stationary environment
$$p(s'|s,a) \text{ is fixed}$$

**Multi-agent reality:** Other agents learning
$$p(s'|s,a_i) = \sum_{a_{-i}} p(s'|s, \mathbf{a}) \pi_{-i}(a_{-i}|s)$$

As $\pi_{-i}$ changes, effective transition dynamics change!

**Consequences:**
- Q-learning may oscillate
- No convergence to Nash equilibrium (in general)
- Can cycle between strategies

### 3.3 Independent Actor-Critic

**Extension to policy gradients:**

Each agent $i$ learns:
- Policy: $\pi_i(a_i|s)$
- Value: $V_i(s)$ or $Q_i(s, a_i)$

**Policy gradient:**
$$\nabla_{\theta_i} J_i = \mathbb{E}[\nabla_{\theta_i} \log \pi_i(a_i|s) Q_i(s, a_i)]$$

Same non-stationarity issues as IQL!

## 4. Centralized Training, Decentralized Execution

### 4.1 CTDE Paradigm

**Key insight:** Use global information during training, local at execution

**Training (centralized):**
- Access to joint observations: $\mathbf{o} = (o_1, ..., o_n)$
- Access to joint actions: $\mathbf{a} = (a_1, ..., a_n)$
- Can use global state $s$
- Learn from full information

**Execution (decentralized):**
- Each agent only uses local observation $o_i$
- Policy: $\pi_i(a_i|o_i)$
- No communication required

**Benefits:**
- Stable training (global perspective)
- Scalable execution (local policies)
- Handles partial observability

### 4.2 MADDPG (Multi-Agent DDPG)

**Lowe et al. (2017)**: Extend DDPG to multi-agent

**Centralized critic:**
$$Q_i(\mathbf{o}, \mathbf{a})$$
Takes joint observations and actions

**Decentralized actor:**
$$\pi_i(a_i|o_i)$$
Only uses local observation

**Training:**
- Critic update (with full information):
$$L_i = \mathbb{E}[(r_i + \gamma Q_i'(\mathbf{o}', \mathbf{a}') - Q_i(\mathbf{o}, \mathbf{a}))^2]$$
where $\mathbf{a}' = (\mu_1'(o_1'), ..., \mu_n'(o_n'))$

- Actor update (using centralized critic):
$$\nabla_{\theta_i} J_i = \mathbb{E}[\nabla_{a_i} Q_i(\mathbf{o}, \mathbf{a})|_{a_i=\mu_i(o_i)} \nabla_{\theta_i} \mu_i(o_i)]$$

**Key advantage:** Reduces non-stationarity during training

### 4.3 COMA (Counterfactual Multi-Agent)

**Foerster et al. (2018)**: Actor-critic with counterfactual baselines

**Problem:** Credit assignment in team rewards
- All agents get same reward $r$
- Which agent(s) caused success?

**Solution:** Counterfactual advantage
$$A_i(s, \mathbf{a}) = Q(s, \mathbf{a}) - \sum_{a_i'} \pi_i(a_i'|s) Q(s, (\mathbf{a}_{-i}, a_i'))$$

**Intuition:**
- $Q(s, \mathbf{a})$: Value of joint action
- Baseline: Expected value if agent $i$ acts randomly
- Advantage: How much better is actual action?

**Addresses:** Multi-agent credit assignment

## 5. Value Decomposition Methods

### 5.1 VDN (Value Decomposition Networks)

**Idea:** Decompose team value into individual utilities

$$Q_{tot}(\mathbf{s}, \mathbf{a}) = \sum_{i=1}^n Q_i(s, a_i)$$

**Properties:**
- Additive decomposition
- Decentralized execution: $\arg\max_{a_i} Q_i(s, a_i)$
- Simple, but restrictive (additivity assumption)

### 5.2 QMIX

**Rashid et al. (2018)**: Non-linear value decomposition

$$Q_{tot}(\mathbf{s}, \mathbf{a}) = f(Q_1(o_1, a_1), ..., Q_n(o_n, a_n); s)$$

where $f$ is mixing network with constraint:
$$\frac{\partial Q_{tot}}{\partial Q_i} \geq 0 \quad \forall i$$

**Monotonicity constraint** ensures:
$$\arg\max_{\mathbf{a}} Q_{tot}(\mathbf{s}, \mathbf{a}) = \begin{pmatrix} \arg\max_{a_1} Q_1(o_1, a_1) \\ \vdots \\ \arg\max_{a_n} Q_n(o_n, a_n) \end{pmatrix}$$

**Benefits:**
- More expressive than VDN
- Still allows decentralized execution
- State-of-the-art on StarCraft Multi-Agent Challenge (SMAC)

### 5.3 QTRAN

**Son et al. (2019)**: More general decomposition

Relaxes monotonicity, uses transformation:
$$Q_{tot}(\mathbf{s}, \mathbf{a}) = Q_{jt}(\mathbf{s}, \mathbf{a}) + \sum_i V_i(o_i) - V_{jt}(\mathbf{s})$$

More flexible but more complex

## 6. Communication and Coordination

### 6.1 Communication Protocols

**Types of communication:**
1. **No communication**: Fully decentralized
2. **Implicit**: Learn from observed actions
3. **Explicit**: Send messages

**CommNet (Sukhbaatar et al. 2016):**
- Agents exchange continuous vectors
- Messages averaged across neighborhood
- End-to-end differentiable

$$h_i^{t+1} = f(h_i^t, \frac{1}{|\mathcal{N}_i|} \sum_{j \in \mathcal{N}_i} h_j^t)$$

### 6.2 Emergent Communication

**Can agents learn to communicate?**

**Setup:**
- Agents can send discrete messages
- Messages have no pre-defined meaning
- Learn through RL

**Challenges:**
- Messages are discrete → hard to train
- Need shared understanding ("language")

**Solutions:**
- Gumbel-Softmax trick for discrete messages
- Shared reward to incentivize cooperation

**Observations:**
- Agents develop simple protocols
- Communication emerges when beneficial
- Protocols can be task-specific

### 6.3 Theory of Mind

**Modeling other agents:**

Agent $i$ maintains belief about agent $j$:
$$b_i(\pi_j)$$

**Benefits:**
- Predict others' actions
- Plan accordingly
- Exploit weaknesses (competitive)
- Coordinate better (cooperative)

**Opponent modeling:**
- Learn model of opponent policy
- Use in planning (tree search, MPC)
- Used in Poker AI, AlphaStar

## 7. Self-Play and Population-Based Training

### 7.1 Self-Play

**Train agent by playing against itself:**

```
Initialize policy π
Loop:
  Play games: π vs π
  Collect experience
  Update π
```

**Benefits:**
- Curriculum learning (opponent gets better)
- No need for human data
- Used in: AlphaGo, OpenAI Five, AlphaStar

**Challenges:**
- Forgetting: May forget how to beat old strategies
- Cycling: Can cycle between strategies
- Local optima: May get stuck

### 7.2 League Training

**AlphaStar approach:**

Maintain population of agents:
1. **Main agents**: Training actively
2. **Main exploiters**: Learn to beat main agents
3. **League exploiters**: Learn to beat everyone
4. **Past agents**: Frozen snapshots

**Training:**
- Sample opponents from league
- Prevents forgetting
- Encourages diverse strategies

### 7.3 Policy Space Response Oracles

**Compute approximate Nash equilibrium:**

1. Initialize population $\Pi = \{\pi_1\}$
2. Loop:
   - Compute meta-strategy: $\sigma$ over $\Pi$
   - Train best response to $\sigma$: $\pi_{BR}$
   - Add to population: $\Pi \leftarrow \Pi \cup \{\pi_{BR}\}$

**Convergence:** Approximates Nash equilibrium

## 8. Practical Considerations

### 8.1 Scalability

**Joint action space grows exponentially:**
$$|\mathcal{A}_1 \times ... \times \mathcal{A}_n| = \prod_{i=1}^n |\mathcal{A}_i|$$

**Solutions:**
1. **Factorization**: VDN, QMIX
2. **Mean-field approximation**: Assume symmetric agents
3. **Graph neural networks**: Exploit structure

### 8.2 Partial Observability

**Dec-POMDP**: Decentralized Partially Observable MDP
- Each agent sees local observation $o_i$
- Must act without full state

**Solutions:**
- Recurrent policies (LSTM, GRU)
- Belief state tracking
- Communication

### 8.3 Reward Structures

**Team reward:**
$$r_i = r_{team} \quad \forall i$$
- Full cooperation
- Credit assignment hard

**Individual rewards:**
$$r_i = r_i(s, \mathbf{a})$$
- Captures individual contribution
- May lead to selfish behavior

**Shaped rewards:**
$$r_i = r_{team} + \beta \cdot r_i^{individual}$$
- Balances team and individual goals

## Summary

### Key Takeaways

1. **Multi-agent RL** extends single-agent to multiple learning agents
   - Cooperative, competitive, or mixed settings
   - Non-stationarity is key challenge

2. **Game theory** provides solution concepts:
   - Nash equilibrium
   - Stochastic games (Markov games)
   - Zero-sum vs general-sum

3. **Independent learning** (IQL) is simple but limited:
   - Treats others as environment
   - Non-stationary, no convergence guarantees

4. **CTDE paradigm**: Centralized training, decentralized execution
   - MADDPG: Centralized critic, decentralized actors
   - COMA: Counterfactual credit assignment

5. **Value decomposition**: VDN, QMIX, QTRAN
   - Decompose team value into individual utilities
   - Enable decentralized execution

6. **Communication** can be learned:
   - Emergent protocols
   - Theory of mind

7. **Self-play and leagues** for competitive settings:
   - Curriculum learning
   - Population diversity

8. **Practical challenges**:
   - Scalability (exponential joint space)
   - Partial observability
   - Credit assignment

### Next Steps

- **Lesson 13b**: Implement IQL, MADDPG, VDN, QMIX
- **Lesson 14**: Hierarchical RL
- **Lesson 15**: Meta-learning and transfer

### Further Reading

1. Shoham & Leyton-Brown (2009) - "Multiagent Systems: Algorithmic, Game-Theoretic, and Logical Foundations"
2. Lowe et al. (2017) - "Multi-Agent Actor-Critic for Mixed Cooperative-Competitive Environments" (MADDPG)
3. Foerster et al. (2018) - "Counterfactual Multi-Agent Policy Gradients" (COMA)
4. Rashid et al. (2018) - "QMIX: Monotonic Value Function Factorisation for Decentralised Multi-Agent RL"
5. Vinyals et al. (2019) - "Grandmaster level in StarCraft II using multi-agent RL" (AlphaStar)
6. Berner et al. (2019) - "Dota 2 with Large Scale Deep Reinforcement Learning" (OpenAI Five)

## Exercises

### Theoretical Questions

1. **Prove** that Prisoner's Dilemma has unique Nash equilibrium (Defect, Defect)

2. **Derive** the centralized critic gradient in MADDPG

3. **Show** that VDN decomposition allows decentralized execution

4. **Explain** why QMIX monotonicity constraint enables decentralized execution

5. **Analyze** when communication helps vs hurts coordination

### Computational Exercises

6. **Implement** independent Q-learning on simple matrix game

7. **Visualize** non-stationarity in multi-agent learning curves

8. **Compare** IQL vs MADDPG on cooperative task

9. **Implement** simple CommNet for communication

10. **Train** self-play agent on Tic-Tac-Toe or Connect Four

### Practical Applications

11. **Build** VDN agent for cooperative navigation

12. **Implement** QMIX for StarCraft micro-management

13. **Create** league training system for 2-player game

14. **Design** reward shaping for multi-robot coordination

15. **Analyze** emergent behaviors in multi-agent simulation